In [ ]:
import pandas as pd

In [ ]:

footfall_df = pd.read_csv('../datasets/raw/StationFootfall_2024_2025.csv')

footfall_df.rename(columns={'TravelDate' : 'date', 'DayOfWeek' : 'day_of_week', 'Station' : 'station','EntryTapCount' : 'entry_tap_count', 'ExitTapCount' : 'exit_tap_count'}, inplace=True)

footfall_df['date'] = pd.to_datetime(footfall_df['date'].astype(str), format="%Y%m%d")

footfall_df['baseline_entry'] = (
    footfall_df.groupby(['station', 'day_of_week'])['entry_tap_count']
      .transform('mean')
)

footfall_df['baseline_exit'] = (
    footfall_df.groupby(["station", 'day_of_week'])['exit_tap_count']
      .transform('mean')
)

footfall_df['overcrowding_index'] = (
    footfall_df['entry_tap_count'] / footfall_df['baseline_entry']
) * 100

print(footfall_df.head())

In [ ]:
weather_df = pd.read_csv('../datasets/raw/open-meteo-51.49N0.49W24m.csv')
weather_df['date'] = pd.to_datetime(weather_df['time'], format="%Y-%m-%d")

In [ ]:
df_combined = pd.merge(
    footfall_df,
    weather_df,
    on="date",
    how="left"
)

In [ ]:
df_combined.to_csv('../datasets/raw/combined.csv', index=False)

In [ ]:
df_combined.info()

In [ ]:
df_combined['is_raining'] = (df_combined['rain_sum (mm)'] > 0.0).astype('int')

In [ ]:
df_combined['is_raining']

In [ ]:
df_combined = df_combined.rename(columns={
    "date": "date",
    "day_of_week": "day_of_week",
    "station": "station",
    "entry_tap_count": "entries",
    "exit_tap_count": "exits",
    "baseline_entry": "baseline_entries",
    "baseline_exit": "baseline_exits",
    "overcrowding_index": "overcrowding",
    "time": "hour",
    "temperature_2m_max (°C)": "temp_max",
    "temperature_2m_min (°C)": "temp_min",
    "temperature_2m_mean (°C)": "temp_mean",
    "apparent_temperature_mean (°C)": "app_temp_mean",
    "apparent_temperature_max (°C)": "app_temp_max",
    "apparent_temperature_min (°C)": "app_temp_min",
    "wind_speed_10m_max (km/h)": "wind_max",
    "wind_gusts_10m_max (km/h)": "wind_gust_max",
    "wind_direction_10m_dominant (°)": "wind_dir",
    "rain_sum (mm)": "rain_mm",
    "sunshine_duration (s)": "sunshine_s",
    "daylight_duration (s)": "daylight_s",
    "sunrise (iso8601)": "sunrise",
    "sunset (iso8601)": "sunset",
    "precipitation_hours (h)": "precip_hours",
    "snowfall_sum (cm)": "snow_cm",
    "precipitation_sum (mm)": "precip_mm"
})

In [ ]:
df_combined.to_csv('../datasets/raw/combined.csv', index=False)

In [ ]:
df = pd.read_csv('../datasets/raw/combined.csv')

In [ ]:
# convert date time
df['date'] = pd.to_datetime(df['date'])
df['hour'] = pd.to_datetime(df['hour'], format="%H:%M", errors='coerce').dt.hour

# useful time features
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

# sunrise and sunset
df['sunrise_hour'] = pd.to_datetime(df['sunrise'], errors='coerce').dt.hour
df['sunset_hour'] = pd.to_datetime(df['sunset'], errors='coerce').dt.hour

# encode categories
df['station_code'] = df['station'].astype('category').cat.codes

# numerical day of week
if df['day_of_week'].dtype == object:
    df['day_of_week_code'] = df['day_of_week'].astype('category').cat.codes
else:
    df['day_of_week_code'] = df['day_of_week']


# missing vals
df_rf = df.fillna(0)
df = df_rf.copy()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


# -----------------------------
# 2. Choose your target variable
# -----------------------------
# Predicting entries (you can switch to exit_tap_count or overcrowding_index)
target = "entries"

print('target chosen')

# -----------------------------
# 3. Select features
# -----------------------------
features = df.drop(columns=[
    "date",
    "entries",
    "exits",
    "sunrise",
    "sunset",
    "hour"        # optional: remove if it's a datetime string
], errors="ignore")


X = features
y = df[target]

print('features selected')

# -----------------------------
# 4. Identify categorical columns
# -----------------------------
categorical_cols = ["day_of_week", "station"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# -----------------------------
# 5. Preprocessing
# -----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

print('pre-processing done')

# -----------------------------
# 6. Build the Random Forest pipeline
# -----------------------------
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_split=2,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
        max_samples=0.8
    ))
])

# -----------------------------
# 7. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('test/split done')

# -----------------------------
# 8. Fit the model
# -----------------------------
model.fit(X_train, y_train)

print('model fit')

# -----------------------------
# 9. Evaluate
# -----------------------------
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("MAE:", mae)
print("R²:", r2)

# -----------------------------
# 10. Feature importances
# -----------------------------
rf = model.named_steps["rf"]
ohe = model.named_steps["preprocess"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
all_feature_names = list(cat_feature_names) + numeric_cols

importances = pd.Series(rf.feature_importances_, index=all_feature_names)
print(importances.sort_values(ascending=False).head(20))

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import time

# -----------------------------
# 1️⃣ Load cleaned dataset
# -----------------------------
df = pd.read_csv('../datasets/raw/combined.csv')

# Convert date/time
df['date'] = pd.to_datetime(df['date'])
df['hour'] = pd.to_datetime(df['hour'], format="%H:%M", errors='coerce').dt.hour

# Extract useful time features
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['sunrise_hour'] = pd.to_datetime(df['sunrise'], errors='coerce').dt.hour
df['sunset_hour'] = pd.to_datetime(df['sunset'], errors='coerce').dt.hour

print("time stuff done")

# -----------------------------
# 2️⃣ Encode categorical columns
# -----------------------------
df['station_code'] = df['station'].astype('category').cat.codes
df['day_of_week_code'] = df['day_of_week'].astype('category').cat.codes

print("categorical encoded")

# -----------------------------
# 3️⃣ Fill missing values
# -----------------------------
df = df.fillna(0)

print("missing done")

# -----------------------------
# 4️⃣ Prepare features & target
# -----------------------------
target = 'entries'  # can switch to 'overcrowding' or 'exits'

X = df.drop(columns=['date', 'entries', 'exits', 'station', 'day_of_week', 'sunrise', 'sunset'])
y = df[target]

print("targets and features")

# -----------------------------
# 5️⃣ Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("train/test")

# -----------------------------
# 6️⃣ Build Random Forest
# -----------------------------
rf = RandomForestRegressor(
    n_estimators=100,      # faster, increase later for final model
    max_depth=15,          # limit depth
    min_samples_split=2,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
    max_samples=0.8        # train each tree on 80% of data
)

print("rf done")

# -----------------------------
# 7️⃣ Train & measure time
# -----------------------------
start = time.time()
rf.fit(X_train, y_train)
end = time.time()
print(f"Training time: {end - start:.2f} seconds")

# -----------------------------
# 8️⃣ Evaluate
# -----------------------------
preds = rf.predict(X_test)
print("MAE:", mean_absolute_error(y_test, preds))
print("R²:", r2_score(y_test, preds))

# -----------------------------
# 9️⃣ Feature importance
# -----------------------------
feature_importances = pd.Series(rf.feature_importances_, index=X.columns)
print(feature_importances.sort_values(ascending=False).head(20))


time stuff done
categorical encoded
missing done
targets and features
train/test
rf done
Training time: 32.64 seconds
MAE: 28.311313428982263
R²: 0.9998726223855792
baseline_entries    0.964284
overcrowding        0.033261
baseline_exits      0.002348
station_code        0.000018
day_of_week_code    0.000011
sunshine_s          0.000009
daylight_s          0.000008
day                 0.000008
app_temp_min        0.000007
wind_dir            0.000007
wind_gust_max       0.000005
wind_max            0.000005
month               0.000005
temp_min            0.000004
app_temp_max        0.000003
app_temp_mean       0.000003
precip_hours        0.000002
temp_max            0.000002
rain_mm             0.000002
temp_mean           0.000002
dtype: float64


In [2]:
preds

array([ 2258.61912793,  1022.38519932,  1347.61063745, ...,
        7643.91227006,  9769.47897213, 28657.3493189 ])